In [16]:
import collections

class PuzzleState:
    def __init__(self, board, parent=None, action=None, cost=0):
        self.board = board
        self.parent = parent
        self.action = action
        self.cost = cost
        self.blank_pos = self._find_blank()

    def _find_blank(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return r, c
        return None

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        return hash(str(self.board))

    def __str__(self):
        return '\n'.join([' '.join(map(str, row)) for row in self.board])

    def get_neighbors(self):
        neighbors = []
        r, c = self.blank_pos
        moves = [
            ('up', (-1, 0)),
            ('down', (1, 0)),
            ('left', (0, -1)),
            ('right', (0, 1))
        ]

        for action, (dr, dc) in moves:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                new_board = [list(row) for row in self.board]
                new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
                neighbors.append(PuzzleState(new_board, self, action, self.cost + 1))
        return neighbors

    def get_path(self):
        path = []
        current = self
        while current:
            path.append((current.board, current.action))
            current = current.parent
        return path[::-1]

def depth_limited_search(state, goal_state, limit):
    # Recursive DLS that avoids cycles along the current branch path
    def dls_recursive(node, depth, path_set):
        if node == goal_state:
            return node
        if depth >= limit:
            return None

        for neighbor in node.get_neighbors():
            if neighbor not in path_set:
                path_set.add(neighbor)
                result = dls_recursive(neighbor, depth + 1, path_set)
                if result is not None:
                    return result
                path_set.remove(neighbor) # Backtrack
        return None

    # Initialize path set with the start node
    return dls_recursive(state, 0, {state})

def iterative_deepening_search(initial_state, goal_state, max_depth=25):
    for depth_limit in range(max_depth + 1):
        print(f"Trying depth limit: {depth_limit}")
        result = depth_limited_search(initial_state, goal_state, depth_limit)
        if result:
            return result
    return None

In [17]:
initial_board = [
    [1, 2, 3],
    [4, 0, 5],
    [6, 7, 8]
]

goal_board = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]


initial_state = PuzzleState(initial_board)
goal_state = PuzzleState(goal_board)

print("Initial State:")
print(initial_state)
print("\nGoal State:")
print(goal_state)

print("\nStarting Iterative Deepening Search...")
solution_node = iterative_deepening_search(initial_state, goal_state)

if solution_node:
    print("\nSolution Found!")
    path = solution_node.get_path()
    for i, (board, action) in enumerate(path):
        if i == 0:
            print(f"Step {i}: Initial State")
        else:
            print(f"\nStep {i}: Move '{action}'")
        print('\n'.join([' '.join(map(str, row)) for row in board]))
    print(f"Total steps: {len(path) - 1}")
else:
    print("No solution found within the specified maximum depth.")

Initial State:
1 2 3
4 0 5
6 7 8

Goal State:
1 2 3
4 5 6
7 8 0

Starting Iterative Deepening Search...
Trying depth limit: 0
Trying depth limit: 1
Trying depth limit: 2
Trying depth limit: 3
Trying depth limit: 4
Trying depth limit: 5
Trying depth limit: 6
Trying depth limit: 7
Trying depth limit: 8
Trying depth limit: 9
Trying depth limit: 10
Trying depth limit: 11
Trying depth limit: 12
Trying depth limit: 13
Trying depth limit: 14

Solution Found!
Step 0: Initial State
1 2 3
4 0 5
6 7 8

Step 1: Move 'right'
1 2 3
4 5 0
6 7 8

Step 2: Move 'down'
1 2 3
4 5 8
6 7 0

Step 3: Move 'left'
1 2 3
4 5 8
6 0 7

Step 4: Move 'left'
1 2 3
4 5 8
0 6 7

Step 5: Move 'up'
1 2 3
0 5 8
4 6 7

Step 6: Move 'right'
1 2 3
5 0 8
4 6 7

Step 7: Move 'down'
1 2 3
5 6 8
4 0 7

Step 8: Move 'right'
1 2 3
5 6 8
4 7 0

Step 9: Move 'up'
1 2 3
5 6 0
4 7 8

Step 10: Move 'left'
1 2 3
5 0 6
4 7 8

Step 11: Move 'left'
1 2 3
0 5 6
4 7 8

Step 12: Move 'down'
1 2 3
4 5 6
0 7 8

Step 13: Move 'right'
1 2 3
4 5 6